# CKPT3 Results: Multi-Agent Hallucination Detection

**Project:** Multi-Agent Hallucination Detection in RAG Systems  
**Dataset:** RAGTruth — 600 balanced test samples (200/task)  
**Evaluation:** Track A — Case-level F1, Span-level F1, Bootstrap 95% CI

---
## Methods Compared
| Method | Type | Training? |
|--------|------|-----------|
| RAGTruth LLaMA-2-13B | Supervised fine-tuning | Yes (14K samples) |
| Self-Verification GPT-4o-mini | Zero-shot prompting | No |
| **Multi-Agent Pipeline (ours)** | Zero-shot + NLI | No |

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path('..').resolve()))
from evaluation.metrics import full_evaluation_report, bootstrap_confidence_interval

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Load predictions
gpt_preds = load_jsonl(RESULTS_DIR / 'gpt_baseline_predictions.jsonl')
ma_preds  = load_jsonl(RESULTS_DIR / 'multi_agent_predictions.jsonl')

print(f'Self-Verification predictions: {len(gpt_preds)}')
print(f'Multi-Agent predictions:       {len(ma_preds)}')

## 1. Full Evaluation Reports

In [ ]:
gpt_report = full_evaluation_report(gpt_preds, method_name='Self-Verification GPT-4o-mini')
ma_report  = full_evaluation_report(ma_preds,  method_name='Multi-Agent Pipeline (ours)')

# Official RAGTruth numbers (from Wu et al. 2023 paper)
llama_report = {
    'method': 'RAGTruth LLaMA-2-13B (official)',
    'case_level': {'precision': 0.7380, 'recall': 0.8320, 'f1': 0.7822},
    'per_task': {
        'QA':       {'precision': 0.7200, 'recall': 0.7100, 'f1': 0.7149},
        'Summary':  {'precision': 0.7100, 'recall': 0.7600, 'f1': 0.7341},
        'Data2txt': {'precision': 0.7800, 'recall': 0.9000, 'f1': 0.8358},
    },
    'bootstrap_ci': {'ci_lower': None, 'ci_upper': None},
}

print('Evaluation complete.')
print(f"\nSelf-Verification: F1={gpt_report['case_level']['f1']:.4f}")
print(f"Multi-Agent:       F1={ma_report['case_level']['f1']:.4f}")

## 2. Main Results Table

In [ ]:
def make_row(report):
    cl = report['case_level']
    pt = report.get('per_task', {})
    ci = report.get('bootstrap_ci', {})
    ci_str = f"[{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}]" if ci.get('ci_lower') else 'N/A (cited)'
    return {
        'Method': report['method'],
        'Precision': round(cl['precision'], 4),
        'Recall':    round(cl['recall'], 4),
        'Overall F1': round(cl['f1'], 4),
        'QA F1':      round(pt.get('QA', {}).get('f1', 0), 4),
        'Summary F1': round(pt.get('Summary', {}).get('f1', 0), 4),
        'Data2txt F1':round(pt.get('Data2txt', {}).get('f1', 0), 4),
        '95% CI':     ci_str,
    }

rows = [make_row(llama_report), make_row(gpt_report), make_row(ma_report)]
results_df = pd.DataFrame(rows)
print('\nTable 1: Hallucination Detection Results (Track A, n=600)')
print(results_df.to_string(index=False))

## 3. F1 Comparison Bar Chart

In [ ]:
tasks = ['Overall', 'QA', 'Summary', 'Data2txt']
methods = ['LLaMA-2-13B\n(official)', 'Self-Verification\nGPT-4o-mini', 'Multi-Agent\n(ours)']
colors = ['#4C72B0', '#DD8452', '#55A868']

llama_f1s = [
    llama_report['case_level']['f1'],
    llama_report['per_task']['QA']['f1'],
    llama_report['per_task']['Summary']['f1'],
    llama_report['per_task']['Data2txt']['f1'],
]
gpt_f1s = [
    gpt_report['case_level']['f1'],
    gpt_report['per_task'].get('QA', {}).get('f1', 0),
    gpt_report['per_task'].get('Summary', {}).get('f1', 0),
    gpt_report['per_task'].get('Data2txt', {}).get('f1', 0),
]
ma_f1s = [
    ma_report['case_level']['f1'],
    ma_report['per_task'].get('QA', {}).get('f1', 0),
    ma_report['per_task'].get('Summary', {}).get('f1', 0),
    ma_report['per_task'].get('Data2txt', {}).get('f1', 0),
]

x = np.arange(len(tasks))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - w, llama_f1s, w, label=methods[0], color=colors[0])
b2 = ax.bar(x,     gpt_f1s,   w, label=methods[1], color=colors[1])
b3 = ax.bar(x + w, ma_f1s,    w, label=methods[2], color=colors[2])

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(tasks, fontsize=12)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.set_title('Figure 7: F1 Comparison by Method and Task (Track A, n=600)')
ax.legend(fontsize=10)
ax.axhline(y=gpt_report['case_level']['f1'], color=colors[1], linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, report, title in zip(
    axes,
    [gpt_report, ma_report],
    ['Self-Verification GPT-4o-mini', 'Multi-Agent Pipeline (ours)']
):
    cm = np.array(report['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: Clean', 'Pred: Halluc'],
                yticklabels=['True: Clean', 'True: Halluc'])
    f1 = report['case_level']['f1']
    ax.set_title(f'{title}\nF1={f1:.4f}', fontsize=11)

fig.suptitle('Figure 8: Confusion Matrices (Track A, n=600)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Bootstrap Confidence Intervals

In [ ]:
gpt_ci = gpt_report['bootstrap_ci']
ma_ci  = ma_report['bootstrap_ci']

print('Bootstrap 95% Confidence Intervals (1000 iterations):')
print(f"  Self-Verification: F1={gpt_report['case_level']['f1']:.4f}  "
      f"CI=[{gpt_ci['ci_lower']:.4f}, {gpt_ci['ci_upper']:.4f}]")
print(f"  Multi-Agent:       F1={ma_report['case_level']['f1']:.4f}  "
      f"CI=[{ma_ci['ci_lower']:.4f}, {ma_ci['ci_upper']:.4f}]")

# Visualise CIs
fig, ax = plt.subplots(figsize=(8, 4))
method_labels = ['Self-Verification\nGPT-4o-mini', 'Multi-Agent\n(ours)']
f1s  = [gpt_report['case_level']['f1'], ma_report['case_level']['f1']]
lows = [gpt_ci['ci_lower'], ma_ci['ci_lower']]
highs= [gpt_ci['ci_upper'], ma_ci['ci_upper']]
yerr_low  = [f1 - lo for f1, lo in zip(f1s, lows)]
yerr_high = [hi - f1 for hi, f1 in zip(highs, f1s)]

ax.bar(method_labels, f1s, color=['#DD8452', '#55A868'], width=0.4,
       yerr=[yerr_low, yerr_high], capsize=8, error_kw={'linewidth': 2})
for i, (f, lo, hi) in enumerate(zip(f1s, lows, highs)):
    ax.text(i, hi + 0.01, f'{f:.4f}\n[{lo:.3f}, {hi:.3f}]',
            ha='center', fontsize=10)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.set_title('Figure 9: F1 with 95% Bootstrap Confidence Intervals')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ablation Study

In [ ]:
ABLATION_DIR = RESULTS_DIR / 'ablation'
ablation_results = {}

config_labels = {
    'A_claim_only':    'A: Claim Extractor only',
    'B_nli_full':      'B: NLI on full response',
    'C_no_rules':      'C: A1+A2, majority vote',
    'D_full_pipeline': 'D: Full pipeline (A1+A2+A3)',
}

from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for fname, label in config_labels.items():
    path = ABLATION_DIR / f'ablation_{fname}.jsonl'
    if not path.exists():
        print(f'  [MISSING] {path.name} — run pipeline/run_ablation.py --all first')
        continue
    records = load_jsonl(path)
    y_true = [1 if len(r.get('labels', [])) > 0 else 0 for r in records]
    y_pred = [1 if len(r.get('pred', [])) > 0 else 0 for r in records]
    rows.append({
        'Config': label,
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'n': len(records),
    })
    ablation_results[fname] = records

if rows:
    abl_df = pd.DataFrame(rows)
    print('\nTable 2: Ablation Study Results')
    print(abl_df.to_string(index=False))
else:
    print('No ablation results yet. Run: python pipeline/run_ablation.py --all')

## 7. Error Analysis

In [ ]:
# Categorize errors from multi-agent predictions
false_positives = [r for r in ma_preds
                   if len(r.get('labels', [])) == 0 and len(r.get('pred', [])) > 0]
false_negatives = [r for r in ma_preds
                   if len(r.get('labels', [])) > 0 and len(r.get('pred', [])) == 0]
true_positives  = [r for r in ma_preds
                   if len(r.get('labels', [])) > 0 and len(r.get('pred', [])) > 0]
true_negatives  = [r for r in ma_preds
                   if len(r.get('labels', [])) == 0 and len(r.get('pred', [])) == 0]

print(f'True Positives  (correctly detected): {len(true_positives)}')
print(f'True Negatives  (correctly clean):    {len(true_negatives)}')
print(f'False Positives (false alarm):         {len(false_positives)}')
print(f'False Negatives (missed hallucination):{len(false_negatives)}')

print('\n--- Sample False Negatives (missed hallucinations) ---')
for ex in false_negatives[:3]:
    agg = ex.get('aggregation', {})
    print(f"\nTask: {ex['task_type']} | Model: {ex['model']}")
    print(f"  Gold spans: {[l['text'][:60] for l in ex['labels'][:2]]}")
    print(f"  Verdict counts: {agg.get('verdict_counts', {})}")
    print(f"  Reason system decided clean: {agg.get('reason', 'unknown')}")

print('\n--- Sample False Positives (false alarms) ---')
for ex in false_positives[:3]:
    agg = ex.get('aggregation', {})
    print(f"\nTask: {ex['task_type']} | Model: {ex['model']}")
    print(f"  Predicted spans: {ex.get('pred', [])[:2]}")
    print(f"  Verdict counts: {agg.get('verdict_counts', {})}")

## 8. Hallucination Type Analysis

In [ ]:
label_types = ['Evident Conflict', 'Evident Baseless Info', 'Subtle Conflict', 'Subtle Baseless Info']

def detection_by_type(records):
    """For each hallucination type, how often did we detect it?"""
    detected = Counter()
    total = Counter()
    for r in records:
        if not r.get('labels'):
            continue
        pred_correct = len(r.get('pred', [])) > 0
        for label in r['labels']:
            lt = label.get('label_type', 'Unknown')
            total[lt] += 1
            if pred_correct:
                detected[lt] += 1
    return detected, total

gpt_det, gpt_tot = detection_by_type(gpt_preds)
ma_det,  ma_tot  = detection_by_type(ma_preds)

print('Detection rate by hallucination type:')
print(f'{"Type":<28} {"GPT Self-Verif":>15} {"Multi-Agent":>12}')
print('-' * 58)
for lt in label_types:
    g = gpt_det[lt]/gpt_tot[lt]*100 if gpt_tot[lt] else 0
    m = ma_det[lt]/ma_tot[lt]*100 if ma_tot[lt] else 0
    print(f'{lt:<28} {g:>12.1f}%  {m:>10.1f}%  (n={gpt_tot[lt]})')

## 9. Classification Reports

In [ ]:
print('Self-Verification GPT-4o-mini:')
print(gpt_report['classification_report'])
print('\nMulti-Agent Pipeline (ours):')
print(ma_report['classification_report'])

## 10. Log Run to Experiment Logger

In [ ]:
from logs.experiment_logger import ExperimentLogger

logger = ExperimentLogger()

# Log multi-agent run
logger.log_run(
    method='Multi-Agent Pipeline (GPT-4o-mini + DeBERTa)',
    metrics={
        'n_samples': len(ma_preds),
        'overall_f1': ma_report['case_level']['f1'],
        'overall_precision': ma_report['case_level']['precision'],
        'overall_recall': ma_report['case_level']['recall'],
        'per_task': {
            t: ma_report['per_task'].get(t, {})
            for t in ['QA', 'Summary', 'Data2txt']
        },
        'bootstrap_ci': ma_report['bootstrap_ci'],
    },
    config={'model_gpt': 'gpt-4o-mini', 'model_nli': 'deberta-v3-large', 'seed': 42},
    predictions_file='results/multi_agent_predictions.jsonl',
    notes='Full pipeline on test_balanced.jsonl (600 samples)'
)

# Print summary table
logger.print_summary()